In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from IPython.display import display # Για καλύτερη εμφάνιση των πινάκων
from yellowbrick.cluster import KElbowVisualizer # Παραμένει για το KMeans Elbow

df = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx')
df2 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
df3 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name = 'Loyalty')


#Read the Excel file
#df = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')
#df2 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
#df3 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Loyalty')

print(df.head())

In [ ]:
# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = (
        df2[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    df2[col] = df2[col].replace(["Nan", "None", "Na", ""], np.nan)

df2["CustomCategory"] = df2["Category B"].copy()
# =========================================================
# 3️⃣ ΟΛΕΣ ΟΙ ΠΑΛΙΕΣ ΑΛΛΑΓΕΣ ΣΟΥ ΠΑΝΩ ΣΤΗΝ CustomCategory
# =========================================================

# 3.1 Συσκευασμενο → "Category C + ' σε συσκευασία'"
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = (
    df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
)

# 3.2 Merge γαλακτοκομικών σε ενιαία κατηγορία (θα το σπάσουμε μετά)
to_merge_dairy = [
    "Γιαουρτια σε συσκευασία",
    "Τυροκομικα σε συσκευασία",
    "Γαλατα σε συσκευασία",
    "Βουτυρα σε συσκευασία",
    "Κρεμα Γαλακτος σε συσκευασία",
]
df2["CustomCategory"] = df2["CustomCategory"].replace(
    to_merge_dairy, "Γαλακτοκομικά σε συσκευασία"
)

# 3.3 Ρουχων + Ενδυση → Ρούχα & Ενδυση
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση"
)

# 3.4 Μπυρες + Κρασια + Οινοπνευματωδη → Οινοπνευματωδη
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη"
)

# 3.5 Σωματος / Ξυριστικα / Χεριων / Προσωπου → Προϊόντα Προσωπικής Φροντίδας
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"],
    "Προϊόντα Προσωπικής Φροντίδας",
)

# 3.6 Βαμβακια / Πανες Ακρατειας → Προιοντα Χαρτου
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου"
)

# 3.7 Μωρομαντηλα / Πανες Παιδικες → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μωρομαντηλα", "Πανες Παιδικες"], "Παιδικα"
)

# 3.8 Βρεφικη Τροφη → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Βρεφικη Τροφη", "Παιδικα"
)

# 3.9 Χυμοι / Ροφηματα → Χυμοί & Ροφήματα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία"],
    "Χυμοί & Ροφήματα",
)
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Αναψυκτικα", "Χυμοί & Ροφήματα"
)

# 3.10 Κρεας σε συσκευασία → Κατεψυγμενα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Κρεας σε συσκευασία", "Κατεψυγμενα"
)

# 3.11 Σαλτσες / Dressings → Σάλτσες & Dressings
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σαλτσες", "Dressings"], "Σάλτσες & Dressings"
)

# =========================================================
# 4️⃣ ΟΛΕΣ ΟΙ ΝΕΕΣ "ΕΞΥΠΝΕΣ" ΑΛΛΑΓΕΣ ΑΠΟ ΤΗΝ ΑΝΑΛΥΣΗ
# =========================================================

# 4.1 Split Ρούχα & Ενδυση → Προϊόντα Πλυντηρίου Ρούχων vs Ρούχα
laundry_items = [
    "Υγρα Πλυντηριου",
    "Μαλακτικα Πλυντηριου",
    "Ενισχυτικα-Χρωμοπαγιδες",
    "Σκονη Πλυντηριου",
    "Ταμπλετες Πλυντηριου",
    "Αποσκληρυντικα Πλυντηριου",
    "Πλυσιμο Στο Χερι",
    "Σιδερωματος",
]

mask_laundry = (
    (df2["CustomCategory"] == "Ρούχα & Ενδυση")
& (df2["Category C"].isin(laundry_items))
)
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"

# 4.2 Διάλυση "Χυμα" σε λογικές κατηγορίες
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία",
    "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία",
    "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι",
    "Αλιπαστα": "Κονσερβες",
    "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",
}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = (
    df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
)

# 4.3 Αλευρι από Αρτοσκευασματα → Βασικά Υλικά Μαγειρικής
mask_alevri = (df2["Category B"] == "Αρτοσκευασματα") & (df2["Category C"] == "Αλευρι")
df2.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"

# 4.4 Σπάσιμο Γαλακτοκομικών σε επιμέρους κατηγορίες
dairy_split_map = {
    "Γιαουρτια": "Γιαουρτια",
    "Τυροκομικα": "Τυροκομικα",
    "Γαλατα": "Γαλατα",
    "Βουτυρα": "Βουτυρα",
    "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",
}
mask_dairy = df2["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
df2.loc[mask_dairy, "CustomCategory"] = (
    df2.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")
)

# 4.5 Split Γλυκα Σνακ
mask_glyka = df2["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = df2["Category C"]

df2.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
df2.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
df2.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
df2.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
df2.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
df2.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"

# 4.6 Split Πρωινο
mask_proino = df2["CustomCategory"] == "Πρωινο"
c_pro = df2["Category C"]

# Ροφήματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]),
         "CustomCategory"] = "Ροφηματα Πρωινου"

# Δημητριακά
df2.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"

# Αλείμματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]),
         "CustomCategory"] = "Αλειμματα Πρωινου"

# Εβαπορε → Γαλατα
df2.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"

# 4.7 Split Χυμοί & Ροφήματα
mask_drinks = df2["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = df2["Category C"]

# Αναψυκτικά
df2.loc[mask_drinks & c_dr.isin(
    ["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]
), "CustomCategory"] = "Αναψυκτικα"

# Χυμοί & Νέκταρ
df2.loc[mask_drinks & c_dr.isin(
    ["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]
), "CustomCategory"] = "Χυμοι & Νεκταρ"

# Έτοιμα ροφήματα (Ice Tea, Ice Coffee κ.λπ.)
df2.loc[mask_drinks & c_dr.isin(
    ["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]
), "CustomCategory"] = "Rtd Ροφηματα"

# Ενεργειακά
df2.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"

# 4.8 Split Κατεψυγμενα
mask_frozen = df2["CustomCategory"] == "Κατεψυγμενα"
c_fr = df2["Category C"]

df2.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
df2.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
df2.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
df2.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
df2.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"

# 4.9 Split Αλμυρα Σνακ → Ξηροι Καρποι ξεχωριστά
mask_salty = df2["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = df2["Category C"]

df2.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"

# =========================================================
# 5️⃣ Κανόνας: μικρές κατηγορίες (<10 barcodes) → "Διαφορα"
# =========================================================
counts = df2["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
df2["CustomCategory"] = df2["CustomCategory"].replace(small_cats, "Διαφορα")

# Ζυμες Ψυγειου σε συσκευασία → Αρτοσκευασματα
df2.loc[df2["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"

# =========================================================
# 6️⃣ Γρήγορος έλεγχος
# =========================================================
print("Μοναδικές Category B       :", df2["Category B"].nunique())
print("Μοναδικές CustomCategory   :", df2["CustomCategory"].nunique())
print("\nTop 40 CustomCategory:")
print(df2["CustomCategory"].value_counts().head(40))

# Νέα ενότητα

In [ ]:
#exclude quantities < 1
print(df.shape)
df = df[df['Quantity'] >= 1]
print(df.shape)

In [ ]:
display(df.describe())

In [ ]:
#exclude non positive values
df = df[df['Value_'] > 0]
print(df.shape)

In [ ]:
#exclude non integer quantities
df = df[df['Quantity'] % 1 == 0]
print(df.shape)

In [ ]:
#create new column Price
df['Price'] = df['Value_'] / df['Quantity']
print(df.shape)

In [ ]:
#Remove baskets with no LoyaltyCard attached
df = df[df['LoyaltyCard_ID'].notna()]
print(df.shape)


In [ ]:
df = df[df['Value_'].notna()]
df = df[df['Barcode'].notna()]
df = df[df['Date_'].notna()]
df = df[df['Basket_ID'].notna()]
df = df[df['Quantity'].notna()]
print(df.shape)
df.head()

In [ ]:
#Exclude barcodes that are not contained in the list of real barcodes
df = df[df['Barcode'].isin(df2['Barcode'])]
print(df.shape)

In [ ]:
#Exclude transactions that did not contain cardid's where cardholder was known or his Status was na
# try to fix the na into NaN?
#valid_cards = df3.loc[df3['Status'].str.contains('na', na=False), 'Cardholder']

#df = df[~df['LoyaltyCard_ID'].isin(valid_cards)]Fsi
#print(df.shape)

In [ ]:
#Convert date from string to datetime
df['Date_'] = pd.to_datetime(df['Date_'], errors='coerce', dayfirst=True)
df.head()

In [ ]:
df.replace(['na'], pd.NA, inplace=True)
df.describe()


In [ ]:
Q1 = df['Value_'].quantile(0.25)
Q3 = df['Value_'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
df = df[(df['Value_'] >= lower_bound) & (df['Value_'] <= upper_bound)]
print(df.shape)

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
df.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()

In [ ]:
# [ΠΕΡΙΠΟΥ ΓΡΑΜΜΗ 160]
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()
# --- ΚΑΘΑΡΟ MERGE DF ΜΕ CUSTOM CATEGORY ---
# Προετοιμάζουμε το df_clean κρατώντας μόνο τις βασικές στήλες συναλλαγής
df.rename(columns={"Value_": "Value"}, inplace=True) 
df_clean = df[['Basket_ID', 'Value', 'Quantity', 'Barcode']].copy()

# Εισάγουμε μόνο τις στήλες Barcode και CustomCategory από το df2
df_clean = pd.merge(df_clean, df2[['Barcode', 'CustomCategory']], on='Barcode', how='inner')

# Αφαίρεση NaT/NaN
df_clean.dropna(subset=['CustomCategory', 'Basket_ID', 'Value'], inplace=True)
print(f"Διαστάσεις DataFrame μετά το Clean/Merge: {df_clean.shape}")


In [ ]:
# [ΑΦΑΙΡΕΣΤΕ ΤΟ ΠΡΟΗΓΟΥΜΕΝΟ ΚΑΙ ΒΑΛΤΕ ΑΥΤΟ]
# ---------------------------------------------------------------------------
# --- ΔΗΜΙΟΥΡΓΙΑ final_basket_df (K-Means Features) ---

# 1. Υπολογισμός ποσοτικών χαρακτηριστικών ανά καλάθι
# Χρησιμοποιούμε το df_clean, το οποίο δημιουργήθηκε με το ασφαλές merge και περιέχει CustomCategory
df_to_feature = df_clean.copy() 

basket_feats = df_to_feature.groupby("Basket_ID").agg(
    Total_Value=("Value", "sum"),
    Total_Quantity=('Quantity', 'sum'),
    Unique_Items=("Barcode", "nunique")
)

# 2. Υπολογισμός μεριδίων κατηγοριών (Category Share)
cat_value = df_to_feature.groupby(["Basket_ID", "CustomCategory"])["Value"].sum().unstack(fill_value=0)
cat_share = cat_value.div(cat_value.sum(axis=1).replace(0, 1), axis=0)
cat_share.columns = [f"Share_{c}" for c in cat_share.columns]

# 3. Ενοποίηση σε final_basket_df
final_basket_df = basket_feats.join(cat_share, how="inner").fillna(0)
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

print("\nfinal_basket_df created successfully (using df_clean).")
print(f"Features for K-Means: {final_basket_df.shape[1]} (3 Base + {len(share_cols)} Shares)")
# display(final_basket_df.head()) # Κρατήστε το display αν θέλετε να το δείτε

# --- 3. K-Means Clustering και K-Selection (Με εμφάνιση τιμών k) ---

# 1. Προετοιμασία Δεδομένων & Standardization
X_base = final_basket_df[["Total_Value", "Total_Quantity", "Unique_Items"]].copy()
X_shares = final_basket_df[share_cols].copy()

scaler_base = StandardScaler()
X_base_scaled = scaler_base.fit_transform(X_base)
scaler_shares = StandardScaler()
X_shares_scaled = scaler_shares.fit_transform(X_shares)

# 2. Μείωση Διαστατικότητας (PCA)
pca = PCA(n_components=0.80, random_state=0).fit(X_shares_scaled)
X_pca = pca.transform(X_shares_scaled)
print(f"\nPCA: {X_shares_scaled.shape[1]} features reduced to {X_pca.shape[1]} components (80% Variance).")

X_basket_kmeans = np.hstack([X_base_scaled, X_pca])
# --- 1. Προετοιμασία Δεδομένων & Standardization (Αυτό υποτίθεται ότι έχει τρέξει ήδη) ---
X_base = final_basket_df[["Total_Value", "Total_Quantity", "Unique_Items"]].copy()
X_shares = final_basket_df[share_cols].copy()

scaler_base = StandardScaler()
X_base_scaled = scaler_base.fit_transform(X_base)
scaler_shares = StandardScaler()
X_shares_scaled = scaler_shares.fit_transform(X_shares)

pca = PCA(n_components=0.80, random_state=0).fit(X_shares_scaled)
X_pca = pca.transform(X_shares_scaled)
X_basket_kmeans = np.hstack([X_base_scaled, X_pca])

K_range = range(2, 11)
inertias = []
sil_scores = []

print("\n" + "="*80)
print("--- K-SELECTION FOR BASKET K-MEANS ---")
print(f"{'k':<5} {'Inertia':<15} {'Silhouette Score':<20}")
print("-" * 40)

for k in K_range:
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3, batch_size=1024)
    kmeans.fit(X_basket_kmeans)
    
    inertias.append(kmeans.inertia_)
    
    sample_size = min(5000, X_basket_kmeans.shape[0])
    sil = silhouette_score(X_basket_kmeans, kmeans.labels_, sample_size=sample_size, random_state=42)
    sil_scores.append(sil)
    
    print(f"{k:<5} {inertias[-1]:<15.4f} {sil_scores[-1]:<20.6f}")

# 4. Τελική Εκτέλεση (Βασισμένη στο Max Silhouette Score: k=6)
if len(sil_scores) > 0:
    best_idx = int(np.argmax(sil_scores))
    k_final_basket = list(K_range)[best_idx]
    print(f"\nΣύμφωνα με το Silhouette Score, το βέλτιστο k είναι: {k_final_basket}")
else:
    k_final_basket = 6
    print(f"\nΕπιλέγουμε k={k_final_basket} ως default.")


print(f"\n--- Running Final K-Means with k={k_final_basket} ---")
kmeans_basket = KMeans(n_clusters=k_final_basket, random_state=42, n_init=10).fit(X_basket_kmeans)

final_basket_df["Basket_KMeans_Cluster"] = kmeans_basket.labels_

print("\nBasket Clustering Complete!")
print(f"Κατανομή Clusters:\n{final_basket_df['Basket_KMeans_Cluster'].value_counts().sort_index()}")
print("========================================================")

# 5. Ανάλυση Προφίλ
cluster_profiles_kmeans = final_basket_df.groupby('Basket_KMeans_Cluster').mean()

print("\n--- Μέσες τιμές Ποσοτικών Features ανά Cluster ---")
print(cluster_profiles_kmeans[['Total_Value', 'Total_Quantity', 'Unique_Items']].round(2).to_markdown())

# 6. Top Shares
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

print("\n--- Top 3 Category Shares ανά Cluster ---")
for cluster in cluster_profiles_kmeans.index:
    top_shares = cluster_profiles_kmeans.loc[cluster, share_cols].sort_values(ascending=False).head(3)
    print(f"Cluster {cluster}:")
    print(top_shares.round(3).to_markdown())
    print("-" * 15)

In [ ]:
# --- ΑΝΑΛΥΣΗ ΜΕΡΙΔΙΩΝ ΚΑΤΗΓΟΡΙΩΝ (Category Shares) ---
# Βεβαιωθείτε ότι η στήλη 'share_cols' έχει οριστεί.
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

print("\n--- Top 3 Category Shares ανά Cluster ---")
for cluster in cluster_profiles_kmeans.index:
    top_shares = cluster_profiles_kmeans.loc[cluster, share_cols].sort_values(ascending=False).head(3)
    print(f"Cluster {cluster}:")
    print(top_shares.round(3).to_markdown())
    print("-" * 15)

In [ ]:
# --- ΑΝΑΛΥΣΗ ΜΕΡΙΔΙΩΝ ΚΑΤΗΓΟΡΙΩΝ (Category Shares) ---
# Βεβαιωθείτε ότι η στήλη 'share_cols' έχει οριστεί.
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

print("\n--- Top 3 Category Shares ανά Cluster ---")
for cluster in cluster_profiles_kmeans.index:
    top_shares = cluster_profiles_kmeans.loc[cluster, share_cols].sort_values(ascending=False).head(3)
    print(f"Cluster {cluster}:")
    print(top_shares.round(3).to_markdown())
    print("-" * 15)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- 1. Διασφάλιση ότι το cluster_profiles περιέχει τα shares ---
# Χρησιμοποιούμε το final_basket_df, το οποίο περιέχει όλα τα Shares, 
# και ομαδοποιούμε με τη σωστή στήλη του cluster.

# **ΔΙΟΡΘΩΣΗ ΕΔΩ:** Χρησιμοποιούμε το 'Basket_KMeans_Cluster'
cluster_profiles_with_shares = final_basket_df.groupby('Basket_KMeans_Cluster').mean()

# 2. Καθορισμός Κατηγοριών προς Εξαίρεση
exclude_shares_names = [
    'Τυροκομικα',
    'Γαλατα',
    'Αρτοσκευασματα'
]
exclude_shares = [f'Share_{name}' for name in exclude_shares_names]

# 3. Φιλτράρισμα: Αφαίρεση των Εξαιρετέων Στηλών από το προφίλ
cluster_profiles_filtered = cluster_profiles_with_shares.drop(
    columns=exclude_shares, 
    errors='ignore'
)

# 4. Επιλογή ΜΟΝΟ των Στηλών Shares
# Η λίστα share_cols έχει οριστεί νωρίτερα στον κώδικα.
shares_in_filtered_df = [col for col in cluster_profiles_filtered.columns if col.startswith('Share_')]
cluster_profiles_final_shares = cluster_profiles_filtered[shares_in_filtered_df]

# 5. Ορίζουμε το ΝΕΟ κατώφλι: Μερίδιο Αξίας >= 3%
THRESHOLD = 0.01 

# 6. Δημιουργία Λεξικού με Φιλτραρισμένα Προφίλ
filtered_profiles_dict = {
    cluster: row[row >= THRESHOLD].sort_values(ascending=False)
    for cluster, row in cluster_profiles_final_shares.iterrows()
}

# 7. Δημιουργία Γραφημάτων
print("\n" + "="*80)
print(f"--- Οπτικοποίηση Clusters (Μερίδια Αξίας \u2265 {THRESHOLD*100:.0f}%) ---")
print(f"--- Εξαιρούνται: {', '.join(exclude_shares_names)} ---")
print("="*80)

for cluster, series in filtered_profiles_dict.items():
    if not series.empty:
        plt.figure(figsize=(10, 5))
        
        # Μετατροπή των ονομάτων (αφαίρεση 'Share_')
        series.index = series.index.str.replace('Share_', '')
        
        series.plot(kind='bar', color='darkgreen', edgecolor='black')
        
        plt.title(f"Cluster {cluster}: Προφίλ Μεριδίων Αξίας (Shares \u2265 {THRESHOLD*100:.0f}%)", 
                  fontsize=14, fontweight='bold')
        
        plt.ylabel("Μέσο Μερίδιο Αξίας στο Καλάθι (Average Value Share)", fontsize=10)
        plt.xlabel("Κατηγορία", fontsize=11)
        plt.ylim(0, series.max() * 1.1) 
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.5)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Δεν βρέθηκαν κατηγορίες με Μερίδιο Αξίας \u2265 {THRESHOLD*100:.0f}% για το Cluster {cluster}.")

In [ ]:
!pip install umap-learn
!pip install apyori

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import time

# ====================================================================
# 0. ΟΡΙΣΜΟΣ ΑΡΧΕΙΩΝ & ΠΡΟΕΤΟΙΜΑΣΙΑ
# ====================================================================

# Χρησιμοποιούμε τα ονόματα αρχείων CSV ως strings
FILE_HIER = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
FILE_POS = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')


In [ ]:
# 1. Φόρτωμα & Καθαρισμός Ιεραρχίας
hier = FILE_HIER.copy()  # Χρησιμοποίησε το DataFrame που είναι ήδη φορτωμένο
for col in ["Category A", "Category B", "Category C"]:
    hier[col] = hier[col].astype(str).str.strip().str.title()
    hier[col] = hier[col].replace(["Nan", "None", "Na", ""], np.nan)
hier["CustomCategory"] = hier["Category B"].copy()

# ... (ΟΛΟΙ ΟΙ ΚΑΝΟΝΕΣ CUSTOM CATEGORY) ...
mask_sysk = hier["CustomCategory"] == "Συσκευασμενο"
hier.loc[mask_sysk, "CustomCategory"] = hier.loc[mask_sysk, "Category C"] + " σε συσκευασία"
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία", ]
hier["CustomCategory"] = hier["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"], "Προϊόντα Προσωπικής Φροντίδας")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μωρομαντηλα", "Πανες Παιδικες", "Βρεφικη Τροφη"], "Παιδικα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία", "Αναψυκτικα"], "Χυμοί & Ροφήματα")
hier["CustomCategory"] = hier["CustomCategory"].replace("Κρεας σε συσκευασία", "Κατεψυγμενα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σαλτσες", "Dressings"], "Σάλτσες & Dressings")
laundry_items = ["Υγρα Πλυντηριου", "Μαλακτικα Πλυντηριου", "Ενισχυτικα-Χρωμοπαγιδες", "Σκονη Πλυντηριου", "Ταμπλετες Πλυντηριου", "Αποσκληρυντικα Πλυντηριου", "Πλυσιμο Στο Χερι", "Σιδερωματος", ]
mask_laundry = ((hier["CustomCategory"] == "Ρούχα & Ενδυση") & (hier["Category C"].isin(laundry_items)))
hier.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"
xuma_map = {"Τυροκομικα": "Γαλακτοκομικά σε συσκευασία", "Αλλαντικα": "Αλλαντικα σε συσκευασία", "Μαναβικη": "Μαναβικη σε συσκευασία", "Ξηροι Καρποι": "Αλμυρα Σνακ", "Χαλβας": "Χαλβαδες Ταχινι", "Αλιπαστα": "Κονσερβες", "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",}
mask_xuma = hier["Category B"] == "Χυμα"
hier.loc[mask_xuma, "CustomCategory"] = (hier.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα"))
mask_alevri = (hier["Category B"] == "Αρτοσκευασματα") & (hier["Category C"] == "Αλευρι")
hier.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"
dairy_split_map = {"Γιαουρτια": "Γιαουρτια", "Τυροκομικα": "Τυροκομικα", "Γαλατα": "Γαλατα", "Βουτυρα": "Βουτυρα", "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",}
mask_dairy = hier["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
hier.loc[mask_dairy, "CustomCategory"] = (hier.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία"))
mask_glyka = hier["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = hier["Category C"]
hier.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
hier.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
hier.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
hier.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
hier.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
hier.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"
mask_proino = hier["CustomCategory"] == "Πρωινο"
c_pro = hier["Category C"]
hier.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]), "CustomCategory"] = "Ροφηματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"
hier.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]), "CustomCategory"] = "Αλειμματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"
mask_drinks = hier["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = hier["Category C"]
hier.loc[mask_drinks & c_dr.isin(["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]), "CustomCategory"] = "Αναψυκτικα"
hier.loc[mask_drinks & c_dr.isin(["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]), "CustomCategory"] = "Χυμοι & Νεκταρ"
hier.loc[mask_drinks & c_dr.isin(["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]), "CustomCategory"] = "Rtd Ροφηματα"
hier.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"
mask_frozen = hier["CustomCategory"] == "Κατεψυγμενα"
c_fr = hier["Category C"]
hier.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
hier.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
hier.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
hier.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
hier.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"
mask_salty = hier["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = hier["Category C"]
hier.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"
counts = hier["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
hier["CustomCategory"] = hier["CustomCategory"].replace(small_cats, "Διαφορα")
hier.loc[hier["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"




In [ ]:
# 2. Φόρτωμα POS & Καθαρισμός
df_pos = FILE_POS.copy()  # Χρησιμοποίησε το DataFrame αντί CSV
df_pos = df_pos.rename(columns={"Value_": "Value", "Date_": "Date"}, errors="ignore")
df_pos['Date'] = pd.to_datetime(df_pos['Date'], errors='coerce', dayfirst=True)
df_pos = df_pos.merge(hier[["Barcode", "CustomCategory"]], on="Barcode", how="left")
df_pos = df_pos[(df_pos["Quantity"] > 0) & (df_pos["Value"] > 0)].copy()
df_pos = df_pos[df_pos["Quantity"] % 1 == 0]

In [ ]:

# 3. Basket Segmentation (Q1 Features - KMeans on Scaled Data)
basket_feats = df_pos.groupby("Basket_ID").agg(
    Total_Value=("Value", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Unique_Items=("Barcode", "nunique")
)
cat_value = df_pos.groupby(["Basket_ID", "CustomCategory"])["Value"].sum().unstack(fill_value=0)
cat_share = cat_value.div(cat_value.sum(axis=1).replace(0, 1), axis=0)
cat_share.columns = [f"Share_{c}" for c in cat_share.columns]
final_basket_df = basket_feats.join(cat_share, how="left").fillna(0)
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

# PCA & K-Means (K=3 for Baskets)
X_base = StandardScaler().fit_transform(final_basket_df[["Total_Value","Total_Quantity","Unique_Items"]])
X_shares_scaled = StandardScaler().fit_transform(final_basket_df[share_cols])
pca = PCA(n_components=0.80, random_state=0).fit(X_shares_scaled)
X = np.hstack([X_base, pca.transform(X_shares_scaled)])
kmeans_basket = KMeans(n_clusters=6, random_state=42, n_init=10).fit(X)
final_basket_df["Cluster"] = kmeans_basket.labels_



In [ ]:

# 4. Customer Segmentation Features
df_loyal = df_pos.dropna(subset=["LoyaltyCard_ID"])
df_loyal = df_loyal.merge(final_basket_df[["Cluster"]], on="Basket_ID", how="left")

# R: Recency
current_date = df_loyal['Date'].max() + pd.Timedelta(days=1)
recency_df = df_loyal.groupby('LoyaltyCard_ID')['Date'].max().reset_index()
recency_df['Recency'] = (current_date - recency_df['Date']).dt.days

# F, M, Avg Basket Value, Avg Unique Items, Basket Shares
customer_baskets = df_loyal.groupby(['LoyaltyCard_ID', 'Basket_ID']).agg(
    Total_Basket_Value=('Value', 'sum'),
    Cluster=('Cluster', 'first')
).reset_index()

customer_feats = customer_baskets.groupby('LoyaltyCard_ID').agg(
    Frequency=('Basket_ID', 'nunique'),
    Total_Monetary=('Total_Basket_Value', 'sum'),
    Avg_Basket_Value=('Total_Basket_Value', 'mean')
)
customer_feats = customer_feats.join(df_loyal.groupby('LoyaltyCard_ID')['Barcode'].nunique().rename('Avg_Unique_Items_Per_Basket_Total'), how='inner')
customer_feats = customer_feats.join(recency_df.set_index('LoyaltyCard_ID')[['Recency']], how='inner')

cluster_counts = customer_baskets.groupby(['LoyaltyCard_ID', 'Cluster'])['Basket_ID'].nunique().unstack(fill_value=0)
total_baskets = customer_feats['Frequency']
cluster_shares = cluster_counts.div(total_baskets, axis=0)
cluster_shares.columns = [f"Share_Cluster_{c}" for c in sorted(cluster_counts.columns.tolist())]

final_customer_df = customer_feats.join(cluster_shares, how='inner').fillna(0)
final_customer_df = final_customer_df.drop(columns=['Total_Monetary'], errors='ignore') 



In [ ]:
# 5. K-Selection (K=4) - ΜΕ ΚΑΝΟΝΙΚΟΠΟΙΗΣΗ
customer_cols_for_scaling = [
    'Recency', 'Frequency', 'Avg_Basket_Value', 'Avg_Unique_Items_Per_Basket_Total'
] + cluster_shares.columns.tolist()

X_cust = final_customer_df[customer_cols_for_scaling].values

# Κανονικοποίηση σε [0, 1]
from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
X_cust_normalized = scaler_minmax.fit_transform(X_cust)

# ====== PRINT: Έλεγχος κανονικοποίησης ======
print("\n" + "="*80)
print("--- NORMALIZATION CHECK (Min-Max [0, 1]) ---")
print("="*80)
print(f"Shape of normalized data: {X_cust_normalized.shape}")
print(f"\nMin values per feature: {X_cust_normalized.min(axis=0)}")
print(f"Max values per feature: {X_cust_normalized.max(axis=0)}")
print(f"\nFirst 5 rows of normalized data:")
print(pd.DataFrame(X_cust_normalized[:5], columns=customer_cols_for_scaling).round(4))
print(f"\nData statistics after normalization:")
print(pd.DataFrame(X_cust_normalized, columns=customer_cols_for_scaling).describe().round(4))

# Επιπλέον StandardScaler για K-Means
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust_normalized)

print("\n" + "="*80)
print("--- STANDARDIZATION CHECK (StandardScaler) ---")
print("="*80)
print(f"Shape of scaled data: {X_cust_scaled.shape}")
print(f"Mean values per feature (should be ~0): {X_cust_scaled.mean(axis=0).round(6)}")
print(f"Std dev per feature (should be ~1): {X_cust_scaled.std(axis=0).round(6)}")



inertias_cust = []
sil_scores_cust = []
K_range_cust = range(3, 12)

print("\n" + "="*80)
print("--- K-SELECTION ANALYSIS ---")
print("="*80)
print(f"{'k':<5} {'Inertia':<15} {'Silhouette Score':<20}")
print("-" * 40)

for k in K_range_cust:
    mbk_cust = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_cust_scaled)
    inertias_cust.append(mbk_cust.inertia_)
    sample_size = min(5000, X_cust_scaled.shape[0])
    sil = silhouette_score(X_cust_scaled, mbk_cust.labels_, sample_size=sample_size, random_state=42)
    sil_scores_cust.append(sil)
    
    print(f"{k:<5} {inertias_cust[-1]:<15.4f} {sil_scores_cust[-1]:<20.6f}")

if len(sil_scores_cust) > 0:
    best_idx = int(np.argmax(sil_scores_cust))
    k_final_cust = list(K_range_cust)[best_idx]
    print(f"\nBest k by Silhouette: {k_final_cust} (score: {sil_scores_cust[best_idx]:.4f})")
else:
    k_final_cust = 4
    print(f"\nNo silhouette scores available — using fallback k = {k_final_cust}")

print("-" * 40)
print(f"✓ Selected k = {k_final_cust}")
print("="*80)

In [ ]:
# 5.1 Αφαίρεση Outliers (B2B/Internal Accounts)
print("\n" + "="*80)
print("--- OUTLIER DETECTION & REMOVAL ---")
print("="*80)

# Outliers: Frequency > 1000 ή Recency < 2
outlier_mask = (final_customer_df['Frequency'] > 1000) | (final_customer_df['Recency'] < 2)
n_outliers = outlier_mask.sum()
print(f"Detected {n_outliers} outliers (B2B/Internal accounts)")
if n_outliers > 0:
    print("\nOutlier details:")
    print(final_customer_df[outlier_mask][['Recency', 'Frequency', 'Avg_Basket_Value', 'Avg_Unique_Items_Per_Basket_Total']])

# Remove outliers
final_customer_df = final_customer_df[~outlier_mask].copy()
X_cust = final_customer_df[customer_cols_for_scaling].values

# Ξαναδημιουργούμε κανονικοποίηση & standardization
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
X_cust_normalized = scaler_minmax.fit_transform(X_cust)

scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust_normalized)

print(f"\n✓ Outliers removed. New customer count: {len(final_customer_df)}")
print("="*80)

In [ ]:
# ...existing code...
# --- Re-run K-selection AFTER outlier removal ---
inertias_cust = []
sil_scores_cust = []
K_range_cust = range(2, 12)

for k in K_range_cust:
    mbk_cust = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_cust_scaled)
    inertias_cust.append(mbk_cust.inertia_)
    sample_size = min(5000, X_cust_scaled.shape[0])
    sil = silhouette_score(X_cust_scaled, mbk_cust.labels_, sample_size=sample_size, random_state=42)
    sil_scores_cust.append(sil)
    print(f"{k:<5} {inertias_cust[-1]:<15.4f} {sil_scores_cust[-1]:<20.6f}")

if len(sil_scores_cust) > 0:
    best_idx = int(np.argmax(sil_scores_cust))
    k_final_cust = list(K_range_cust)[best_idx]
else:
    k_final_cust = 4

print(f"✓ Selected k = {k_final_cust}")

# --- Safe dynamic cluster names ---
base_names = ["Mainstream", "Stock-Up", "At-Risk", "Premium", "VIP"]
if k_final_cust <= len(base_names):
    cluster_names = {i: base_names[i] for i in range(k_final_cust)}
else:
    cluster_names = {i: (base_names[i] if i < len(base_names) else f"Cluster_{i}") for i in range(k_final_cust)}
# ...existing code...

In [ ]:
# 5.5 UMAP Visualization (ΠΡΟΣΘΗΚΗ)
import umap

print("\n" + "="*80)
print("--- UMAP DIMENSIONALITY REDUCTION ---")
print("="*80)

umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_cust_umap = umap_reducer.fit_transform(X_cust_scaled)

print(f"✓ UMAP transformation complete: {X_cust_umap.shape}")
print("="*80)

In [ ]:
# 5.6 K-Selection Plots (Elbow + Silhouette)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Elbow Curve
axes[0].plot(list(K_range_cust), inertias_cust, '-o', linewidth=2.5, markersize=10, color='steelblue')
axes[0].axvline(x=k_final_cust, color='red', linestyle='--', linewidth=2.5, label=f'Selected k={k_final_cust}')
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method - Customer Segmentation', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)
axes[0].set_xticks(list(K_range_cust))

# Subplot 2: Silhouette Score
axes[1].plot(list(K_range_cust), sil_scores_cust, '-o', linewidth=2.5, markersize=10, color='coral')
axes[1].axvline(x=k_final_cust, color='red', linestyle='--', linewidth=2.5, label=f'Selected k={k_final_cust}')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score - Customer Segmentation', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)
axes[1].set_xticks(list(K_range_cust))

plt.tight_layout()
plt.savefig("customer_k_selection_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ K-Selection plots saved: customer_k_selection_analysis.png")

In [ ]:
# Cell #VSC-5a7565f1 - FIX

# 6. UMAP Visualization με Clusters
kmeans_cust_temp = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
cluster_labels_umap = kmeans_cust_temp.labels_

# ✅ Δυναμικά ονόματα
cluster_names_list = ["Mainstream", "Stock-Up", "At-Risk", "Premium", "VIP"]
cluster_names = {i: (cluster_names_list[i] if i < len(cluster_names_list) else f"Cluster_{i}")
                 for i in range(k_final_cust)}

# UMAP scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_cust_umap[:, 0], X_cust_umap[:, 1], 
                     c=cluster_labels_umap, cmap='viridis', 
                     s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
cbar = plt.colorbar(scatter, label=f'Customer Cluster (k={k_final_cust})')
cbar.set_ticks(range(k_final_cust))
cbar.set_ticklabels([cluster_names.get(i, f"C{i}") for i in range(k_final_cust)])

plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.title(f'Customer Segmentation - UMAP Visualization (k={k_final_cust})', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("customer_umap_clusters.png", dpi=150)
plt.show()

print(f"✓ UMAP Visualization: {len(np.unique(cluster_labels_umap))} clusters detected")
print(f"  Cluster distribution:\n{pd.Series(cluster_labels_umap).value_counts().sort_index()}")

In [ ]:
# Cell #VSC-fe55aed8 - FIX

kmeans_cust = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
final_customer_df["Customer_Cluster"] = kmeans_cust.labels_

# ✅ Δυναμικό aggregation
agg_dict = {
    "Count": ("Customer_Cluster", "size"),
    "Recency_Days_Avg": ("Recency", "mean"),
    "Frequency_Baskets_Avg": ("Frequency", "mean"),
    "Avg_Basket_Value_Avg": ("Avg_Basket_Value", "mean"),
    "Avg_Unique_Items_Total_Avg": ("Avg_Unique_Items_Per_Basket_Total", "mean"),
}

# Προσθήκη Share_Cluster_* στατικά
for col in final_customer_df.columns:
    if col.startswith("Share_Cluster_"):
        agg_dict[f"{col}_Avg"] = (col, "mean")

customer_summary = final_customer_df.groupby("Customer_Cluster").agg(**agg_dict).round(2)

print("\n" + "="*80)
print(f"--- Τελική Περίληψη Customer Segments (k={k_final_cust}) ---")
print("="*80)
print(customer_summary)

In [ ]:
# 7.1 Customer Cluster Profiles Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Plot 1: Cluster Size
cluster_sizes = final_customer_df["Customer_Cluster"].value_counts().sort_index()
axes[0].bar(cluster_sizes.index, cluster_sizes.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Customer Cluster', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].set_title('Cluster Size Distribution', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Average Recency by Cluster
recency_by_cluster = final_customer_df.groupby("Customer_Cluster")["Recency"].mean()
axes[1].bar(recency_by_cluster.index, recency_by_cluster.values, color='coral', edgecolor='black')
axes[1].set_xlabel('Customer Cluster', fontsize=11)
axes[1].set_ylabel('Recency (days)', fontsize=11)
axes[1].set_title('Average Recency by Cluster', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Average Frequency by Cluster
freq_by_cluster = final_customer_df.groupby("Customer_Cluster")["Frequency"].mean()
axes[2].bar(freq_by_cluster.index, freq_by_cluster.values, color='lightgreen', edgecolor='black')
axes[2].set_xlabel('Customer Cluster', fontsize=11)
axes[2].set_ylabel('Frequency (baskets)', fontsize=11)
axes[2].set_title('Average Frequency by Cluster', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

# Plot 4: Average Basket Value by Cluster
basket_val_by_cluster = final_customer_df.groupby("Customer_Cluster")["Avg_Basket_Value"].mean()
axes[3].bar(basket_val_by_cluster.index, basket_val_by_cluster.values, color='gold', edgecolor='black')
axes[3].set_xlabel('Customer Cluster', fontsize=11)
axes[3].set_ylabel('Avg Basket Value (€)', fontsize=11)
axes[3].set_title('Average Basket Value by Cluster', fontsize=12, fontweight='bold')
axes[3].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("customer_cluster_profiles.png", dpi=150)
plt.show()

print("\n✓ Clustering Complete!")
print("✓ Visualizations saved:")
print("  - customer_k_selection_analysis.png")
print("  - customer_umap_clusters.png")
print("  - customer_cluster_profiles.png")

In [ ]:
# ...existing code...
from IPython.display import display, Markdown

# Ensure customer_summary & final_customer_df exist
try:
    summary = customer_summary.copy()
    dfc = final_customer_df.copy()
except NameError:
    print("Error: Εκτέλεσε πρώτα τα cells που φτιάχνουν 'customer_summary' και 'final_customer_df'."); raise

# Add counts if missing
counts = dfc["Customer_Cluster"].value_counts().sort_index()
if "Count" not in summary.columns:
    summary["Count"] = counts

# Show table for verification
print("\n--- Customer summary (per cluster) ---")
display(summary)

# Heuristic labeling (πρόταση — ελέγξτε)
labels = {}
avg_basket = summary["Avg_Basket_Value_Avg"].astype(float)
freq = summary["Frequency_Baskets_Avg"].astype(float)
rec = summary["Recency_Days_Avg"].astype(float)
cnt = summary["Count"].astype(int)

for i in summary.index:
    if cnt.loc[i] <= 3 and freq.loc[i] > summary["Frequency_Baskets_Avg"].max() * 0.8:
        labels[i] = "Outlier / B2B"
    elif avg_basket.loc[i] == avg_basket.max():
        labels[i] = "Stock‑Up (High‑value)"
    elif freq.loc[i] == freq.max():
        labels[i] = "Mainstream (High‑frequency)"
    elif rec.loc[i] == rec.max():
        labels[i] = "At‑Risk / Lapsed"
    else:
        labels[i] = "Premium / Other"

# Display proposed mapping
md = "### Προτεινόμενη αντιστοίχιση cluster → περιγραφή\n\n"
for k, v in labels.items():
    md += f"- Cluster {k}: **{v}**  \n"

display(Markdown(md))
# ...existing code...

In [ ]:
# ...existing code...
mapping = {
    0: "At-Risk / Lapsed",
    1: "Core Mainstream",
    2: "Stock-Up (High-value)",
    3: "High-frequency / Heavy shoppers",
    4: "Premium / Occasional",
    5: "VIP / Elite Customers"
}

# show actual counts + suggested labels
counts = final_customer_df["Customer_Cluster"].value_counts().sort_index()
print("Actual counts:", counts.to_dict())

# print suggested mapping with cluster metrics
for k, label in mapping.items():
    if k in customer_summary.index:
        row = customer_summary.loc[k]
        print(f"Cluster {k}: {label} | Count={int(row['Count'])} | Recency={row['Recency_Days_Avg']} | Freq={row['Frequency_Baskets_Avg']} | AvgBasket={row['Avg_Basket_Value_Avg']}")
    else:
        print(f"Cluster {k}: {label} | (no summary row)")

# Optionally update cluster_names used in plots
cluster_names = {k: v for k, v in mapping.items()}
print("\ncluster_names updated:", cluster_names)
# ...existing code...

In [ ]:
# ====================================================================
# 1. ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ ΓΙΑ MISSION-BASED CLUSTERING
# ====================================================================

# Χρησιμοποιούμε τις κολόνες shares που προέκυψαν από το basket clustering (Share_Cluster_0, κλπ)
cluster_share_cols = [c for c in final_customer_df.columns if c.startswith("Share_Cluster_")]

# Χρησιμοποιούμε όλες τις CustomCategory shares (από το final_basket_df)
# ΠΡΕΠΕΙ να ενσωματώσουμε τις CustomCategory Shares στο final_customer_df

# Βήμα 1: Υπολογισμός μέσων Category Shares ανά πελάτη
# Συνδέουμε το df_loyal (που έχει CustomCategory) με το final_customer_df
# df_loyal = df_pos.dropna(subset=['LoyaltyCard_ID'])
# df_loyal = df_loyal.merge(final_customer_df[['Customer_Cluster']], on='LoyaltyCard_ID', how='inner') # Αυτό έχει γίνει ήδη

# Υπολογίζουμε τις μέσες μετοχές αξίας ανά πελάτη για όλες τις CustomCategories
# Αυτή η προσέγγιση είναι πιο άμεση: Πόσο ξοδεύει ο πελάτης συνολικά σε κάθε κατηγορία.
customer_cat_value = df_loyal.groupby(['LoyaltyCard_ID', 'CustomCategory'])['Value'].sum().unstack(fill_value=0)
customer_total_value = customer_cat_value.sum(axis=1).replace(0, 1)

customer_cat_share = customer_cat_value.div(customer_total_value, axis=0)
customer_cat_share.columns = [f"CS_{c}" for c in customer_cat_share.columns] # CS = Customer Share

# Βήμα 2: Ενοποίηση με το final_customer_df (RFM & Basket Cluster Shares)
final_customer_df_mission = final_customer_df.join(customer_cat_share, how='inner').fillna(0)

# ====================================================================
# 2. ΟΡΙΣΜΟΣ FEATURES ΓΙΑ MISSION K-MEANS
# ====================================================================

# Mission Features: Όλες οι Customer Shares (CS_)
mission_share_cols = [c for c in final_customer_df_mission.columns if c.startswith("CS_")]

# Βασικές Features (Αυτές που βοηθούν στη διάκριση)
base_cols = ['Recency', 'Frequency', 'Avg_Basket_Value'] # Avg_Unique_Items_Per_Basket_Total

mission_cols_for_scaling = base_cols + mission_share_cols

X_mission = final_customer_df_mission[mission_cols_for_scaling].values

# ====================================================================
# 3. SCALING & PCA
# ====================================================================

# Normalization (MinMax)
scaler_minmax_mission = MinMaxScaler(feature_range=(0, 1))
X_mission_normalized = scaler_minmax_mission.fit_transform(X_mission)

# Standardization
scaler_mission = StandardScaler()
X_mission_scaled = scaler_mission.fit_transform(X_mission_normalized)

# Dimensionality Reduction (PCA) για τις πολλές Mission Shares
# Εφαρμογή PCA μόνο στις Mission Shares για καλύτερο αποτέλεσμα
X_base_mission_scaled = X_mission_scaled[:, :len(base_cols)]
X_shares_mission_scaled = X_mission_scaled[:, len(base_cols):]

pca_mission = PCA(n_components=0.85, random_state=42).fit(X_shares_mission_scaled)
X_pca_mission = pca_mission.transform(X_shares_mission_scaled)
print(f"\nMission PCA: {X_shares_mission_scaled.shape[1]} features reduced to {X_pca_mission.shape[1]} components (85% Variance).")

X_mission_kmeans = np.hstack([X_base_mission_scaled, X_pca_mission])


# ====================================================================
# 4. K-SELECTION & FINAL CLUSTERING
# ====================================================================

# Επιλέγουμε ένα εύρος k για Mission/Lifestyle (ίσως 5-7 clusters είναι ιδανικό)
K_range_mission = range(4, 10)
sil_scores_mission = []
best_k_mission = 6 # Default Fallback

print("\n" + "="*80)
print("--- K-SELECTION FOR MISSION-BASED K-MEANS ---")
print(f"{'k':<5} {'Silhouette Score':<20}")
print("-" * 25)

for k in K_range_mission:
    mbk_mission = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_mission_kmeans)
    sample_size = min(5000, X_mission_kmeans.shape[0])
    sil = silhouette_score(X_mission_kmeans, mbk_mission.labels_, sample_size=sample_size, random_state=42)
    sil_scores_mission.append(sil)
    print(f"{k:<5} {sil_scores_mission[-1]:<20.6f}")

if len(sil_scores_mission) > 0:
    best_idx_mission = int(np.argmax(sil_scores_mission))
    best_k_mission = list(K_range_mission)[best_idx_mission]
    print(f"\nBest k by Silhouette for Mission: {best_k_mission}")
else:
    print(f"\nUsing default k = {best_k_mission}")

# Τελική εκτέλεση με το βέλτιστο k
kmeans_mission = KMeans(n_clusters=best_k_mission, random_state=42, n_init=10).fit(X_mission_kmeans)
final_customer_df_mission["Mission_Cluster"] = kmeans_mission.labels_

print(f"\n✓ Mission-Based Clustering Complete with k={best_k_mission}!")
print("========================================================")

# 5. ΑΝΑΛΥΣΗ ΚΑΙ ΟΝΟΜΑΤΟΔΟΣΙΑ (MISSION PROFILES)
# ====================================================================

# Εδώ ο κώδικας υπολογίζει το Mission Profiles
# Δημιουργούμε το Mission Profile με τις μέσες τιμές:
mission_profiles = final_customer_df_mission.groupby('Mission_Cluster').agg(
    Count=("Mission_Cluster", "size"),
    Recency_Avg=("Recency", "mean"),
    Frequency_Avg=("Frequency", "mean"),
    Avg_Basket_Value_Avg=("Avg_Basket_Value", "mean"),
    **{f'{col}_Avg': (col, 'mean') for col in mission_share_cols}
)

# ⚠️ ΔΙΟΡΘΩΣΗ 1: Ενημέρωση της λίστας στηλών shares
# Τώρα οι στήλες έχουν το suffix '_Avg'
mission_share_avg_cols = [c for c in mission_profiles.columns if c.startswith("CS_") and c.endswith("_Avg")]


# 5.1 Top Category Shares ανά Mission Cluster
print("\n--- TOP 3 MISSION SHARES (Category Shares) ανά Cluster ---")
cluster_mission_names = {}
for cluster in mission_profiles.index:
    # Βρίσκουμε τις 5 κορυφαίες μετοχές κατηγοριών, χρησιμοποιώντας τη διορθωμένη λίστα στηλών
    top_shares = mission_profiles.loc[cluster, mission_share_avg_cols].sort_values(ascending=False).head(5)
    
    # Καθαρίζουμε τα ονόματα των στηλών: Αφαιρούμε 'CS_' ΚΑΙ '_Avg'
    top_shares.index = top_shares.index.str.replace('CS_', '').str.replace('_Avg', '')
    
    # Προσπαθούμε να ονομάσουμε το cluster βάσει των κορυφαίων κατηγοριών
    main_interest = top_shares.index[0]
    
    # Λογική Ονοματοδοσίας
    if "Παιδικα" in top_shares.index[:2] and top_shares.index[0] == "Παιδικα":
        name = "Μητέρες / Οικογένειες με Παιδιά"
    elif "Οινοπνευματωδη" in top_shares.index[:2] or "Οινοπνευματωδη" in top_shares.index[2:5] and mission_profiles.loc[cluster, 'Frequency_Avg'] < 5:
        name = "Περιστασιακοί (Events / Πάρτι)"
    elif "Ροφηματα Πρωινου" in top_shares.index[:2] or "Καφες" in top_shares.index[:2]:
        name = "Daily Basics / Καθημερινοί Αγοραστές"
    elif "Βιολογικα" in top_shares.index[:2] or "Τυροκομικα" in top_shares.index[:2]:
        name = "Premium / Health-Conscious"
    elif "Μαναβικη σε συσκευασία" in top_shares.index[:2] and mission_profiles.loc[cluster, 'Avg_Basket_Value_Avg'] < 10:
        name = "Συχνές Μικροαγορές (Γειτονιάς)"
    else:
        name = f"Focus: {main_interest}"

    cluster_mission_names[cluster] = name

    print(f"\nCluster {cluster} - **{cluster_mission_names[cluster]}**:")
    print(f"Count: {mission_profiles.loc[cluster, 'Count']:.0f}, Avg Basket: {mission_profiles.loc[cluster, 'Avg_Basket_Value_Avg']:.2f}€")
    print(top_shares.round(3).to_markdown())
    print("-" * 40)

# ====================================================================
# 6. ΤΕΛΙΚΗ ΣΥΝΟΨΗ & ΟΠΤΙΚΟΠΟΙΗΣΗ
# ====================================================================

mission_summary_table = mission_profiles[['Count', 'Recency_Avg', 'Frequency_Avg', 'Avg_Basket_Value_Avg']].copy()
mission_summary_table['Mission Name'] = mission_summary_table.index.map(cluster_mission_names)

print("\n" + "="*80)
print("--- ΤΕΛΙΚΑ MISSION-BASED CUSTOMER SEGMENTS ---")
print("========================================================")
display(mission_summary_table[['Mission Name', 'Count', 'Recency_Avg', 'Frequency_Avg', 'Avg_Basket_Value_Avg']].to_markdown())

# Plotting the Mission Profiles
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Plot 1: Cluster Size (Mission)
cluster_sizes_mission = final_customer_df_mission["Mission_Cluster"].value_counts().sort_index()
axes[0].bar(cluster_sizes_mission.index, cluster_sizes_mission.values, color='lightcoral', edgecolor='black')
axes[0].set_xlabel('Mission Cluster', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].set_title('Mission Cluster Size Distribution', fontsize=12, fontweight='bold')
axes[0].set_xticks(cluster_sizes_mission.index)
axes[0].set_xticklabels([cluster_mission_names.get(i, f"C{i}") for i in cluster_sizes_mission.index], rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Average Recency by Cluster (Mission)
recency_by_cluster_mission = mission_summary_table["Recency_Avg"]
axes[1].bar(recency_by_cluster_mission.index, recency_by_cluster_mission.values, color='skyblue', edgecolor='black')
axes[1].set_xlabel('Mission Cluster', fontsize=11)
axes[1].set_ylabel('Recency (days)', fontsize=11)
axes[1].set_title('Average Recency by Mission Cluster', fontsize=12, fontweight='bold')
axes[1].set_xticks(recency_by_cluster_mission.index)
axes[1].set_xticklabels([cluster_mission_names.get(i, f"C{i}") for i in recency_by_cluster_mission.index], rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Average Frequency by Cluster (Mission)
freq_by_cluster_mission = mission_summary_table["Frequency_Avg"]
axes[2].bar(freq_by_cluster_mission.index, freq_by_cluster_mission.values, color='mediumpurple', edgecolor='black')
axes[2].set_xlabel('Mission Cluster', fontsize=11)
axes[2].set_ylabel('Frequency (baskets)', fontsize=11)
axes[2].set_title('Average Frequency by Mission Cluster', fontsize=12, fontweight='bold')
axes[2].set_xticks(freq_by_cluster_mission.index)
axes[2].set_xticklabels([cluster_mission_names.get(i, f"C{i}") for i in freq_by_cluster_mission.index], rotation=45, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')

# Plot 4: Average Basket Value by Cluster (Mission)
basket_val_by_cluster_mission = mission_summary_table["Avg_Basket_Value_Avg"]
axes[3].bar(basket_val_by_cluster_mission.index, basket_val_by_cluster_mission.values, color='gold', edgecolor='black')
axes[3].set_xlabel('Mission Cluster', fontsize=11)
axes[3].set_ylabel('Avg Basket Value (€)', fontsize=11)
axes[3].set_title('Average Basket Value by Mission Cluster', fontsize=12, fontweight='bold')
axes[3].set_xticks(basket_val_by_cluster_mission.index)
axes[3].set_xticklabels([cluster_mission_names.get(i, f"C{i}") for i in basket_val_by_cluster_mission.index], rotation=45, ha='right')
axes[3].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Mission-Based Customer Segmentation Visualized.")

Ανοικτή Ερώτηση 2: Πώς κατανέμεται η αγοραστική δύναμη για τις κορυφαίες 4 κατηγορίες που αυξάνουν την αξία του καλαθιού (Value Lift), μεταξύ των Customer Clusters; Δηλαδή, ποιο Customer Cluster (π.χ. Stock-Up, Heavy Shopper) είναι ο πιο σημαντικός αγοραστής των Βιολογικών και των Κύβων/Χαλβάδων (οι κατηγορίες με τον υψηλότερο Value Lift);

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Υποθέτουμε ότι το df_pos (το καθαρό, merged POS DataFrame) είναι διαθέσιμο
# και το final_customer_df (με τη στήλη 'Customer_Cluster') είναι διαθέσιμο

# 1. Βήμα: Ορισμός των Στρατηγικών Κατηγοριών Υψηλής Αξίας
# Αυτές οι κατηγορίες βρέθηκαν ότι έχουν τον υψηλότερο Value Lift %
# Βιολογικα, Κυβοι, Χαλβαδες Ταχινι, Κατεψυγμενα Κρεας & Γευματα
strategic_categories = [
    'Βιολογικα',
    'Κυβοι',
    'Χαλβαδες Ταχινι',
    'Κατεψυγμενα Κρεας & Γευματα'
]

# 2. Βήμα: Ενοποίηση Customer Cluster με το DataFrame συναλλαγών (df_pos)
# Χρειαζόμαστε το df_pos merged με το LoyaltyCard_ID και μετά το Cluster
# (Χρησιμοποιούμε το df_loyal που δημιουργήθηκε σε προηγούμενα βήματα, το οποίο περιέχει
# τις στήλες 'LoyaltyCard_ID', 'CustomCategory', 'Value' και 'Customer_Cluster')

# Merge df_pos with Customer Clusters (επαναλαμβάνουμε για σιγουριά)
df_loyal = df_pos.dropna(subset=['LoyaltyCard_ID']).merge(
    final_customer_df[['Customer_Cluster']],
    on='LoyaltyCard_ID',
    how='inner'
).copy()

# Filter μόνο για τις στρατηγικές κατηγορίες
df_strategic = df_loyal[df_loyal['CustomCategory'].isin(strategic_categories)].copy()

# 3. Βήμα: Υπολογισμός συνολικής αξίας ανά Κατηγορία και Customer Cluster
value_per_category_cluster = df_strategic.groupby(
    ['Customer_Cluster', 'CustomCategory']
)['Value'].sum().reset_index()

# 4. Βήμα: Υπολογισμός του Global Value Share (Ποιος αγοράζει τι)
# Ποιο Customer Cluster έχει το μεγαλύτερο μερίδιο αξίας για κάθε στρατηγική κατηγορία

# Συνολική αξία ανά στρατηγική κατηγορία
total_value_per_cat = df_strategic.groupby('CustomCategory')['Value'].sum().reset_index().rename(
    columns={'Value': 'Total_Cat_Value'}
)

# Ενοποίηση και υπολογισμός Value Share (μερίδιο του Cluster επί του συνόλου της κατηγορίας)
category_cluster_share = value_per_category_cluster.merge(
    total_value_per_cat, on='CustomCategory', how='left'
)
category_cluster_share['Value_Share'] = (
    category_cluster_share['Value'] / category_cluster_share['Total_Cat_Value']
)

# 5. Βήμα: Δημιουργία Pivot Table για Οπτικοποίηση
pivot_table = category_cluster_share.pivot_table(
    index='CustomCategory',
    columns='Customer_Cluster',
    values='Value_Share',
    fill_value=0
)

# --- Εμφάνιση Πίνακα για Ακρίβεια ---
print("\n" + "="*80)
print(f"--- Customer Cluster Value Share for High-Lift Categories ---")
print("Ποσοστό της συνολικής αξίας της κατηγορίας που αγοράζει κάθε Cluster.")
print("="*80)
print(pivot_table.T.round(3).to_markdown(floatfmt=".1%"))


# 6. Βήμα: Οπτικοποίηση (Stacked Bar Plot)
plt.figure(figsize=(10, 7))

pivot_table.T.plot(
    kind='bar',
    stacked=True,
    colormap='Spectral', # Χρώματα που διαφοροποιούν τα clusters
    ax=plt.gca(),
    edgecolor='black'
)

# --- Βελτίωση Γραφήματος ---
plt.title(
    'Customer Cluster Value Share for High-Value-Lift Categories',
    fontsize=14,
    fontweight='bold'
)
plt.ylabel('Share of Total Category Value', fontsize=12)
plt.xlabel('Customer Cluster', fontsize=12)
plt.xticks(
    ticks=pivot_table.T.index,
    labels=[f"C{i}\n({final_customer_df['Customer_Cluster'].value_counts().get(i, 0)} Count)" for i in pivot_table.T.index],
    rotation=0
)
plt.yticks(np.arange(0, 1.1, 0.1), [f'{i*100:.0f}%' for i in np.arange(0, 1.1, 0.1)])

plt.legend(
    title='High-Lift Category',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show() # 

print("\n✓ Οπτικοποίηση της Στόχευσης Ολοκληρώθηκε.")